# Stable Three-Model Blending: Public Demonstration

Combine aligned synthetic OOF predictions using a finite simplex grid and an intersection of near-optimal weight sets. **Four repetitions of five weight partitions reuse the existing OOF; they do not train twenty new sets of base models.** This demonstration neither contains the competition weights nor reproduces its leaderboard score.


## 1. Load the same panel and verify prediction provenance

Run notebooks 01–03 first. Fingerprint, file-hash, ordered-ID, and finite-value checks prevent stale or positional blends.


In [ ]:
from pathlib import Path
import os
import sys

ROOT = next(
    p for p in (Path.cwd(), *Path.cwd().parents)
    if (p / "pyproject.toml").is_file() and (p / "src" / "quant_portfolio").is_dir()
)
sys.path.insert(0, str(ROOT / "src"))
os.environ.setdefault("OMP_NUM_THREADS", "1")
os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")
ARTIFACT_DIR = Path(os.environ.get("PORTFOLIO_ARTIFACT_DIR", str(ROOT / "artifacts")))
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

import pandas as pd
from quant_portfolio.data import make_synthetic_panel
from quant_portfolio.blend import weighted_directional_accuracy
from quant_portfolio.models import model_oof, refit_predict
from quant_portfolio.artifacts import save_model_artifacts

panel = make_synthetic_panel(seed=42)
print("Independent synthetic data only:", panel.train.shape, panel.test.shape)


In [ ]:
from quant_portfolio.artifacts import load_prediction_stack
from quant_portfolio.blend import learn_common_weights, cross_fitted_weight_score
from quant_portfolio.demo import run_ensemble

oof, test_raw = load_prediction_stack(ARTIFACT_DIR, panel)
display(oof.head())
display(pd.Series({
    name: weighted_directional_accuracy(panel.y, oof[name])
    for name in oof.columns
}, name="synthetic_oof_selection_score"))


## 2. Intersect near-optimal weight sets

For each of 20 training-four-fold partitions, find the best score on the evaluated grid. A common candidate loses at most epsilon relative to each partition's best. Then maximize full OOF within that common set, with a deterministic tie-break.

The public grid step (0.05) and tolerance (0.05 score units) are deliberately coarse demonstration settings, not the private competition settings. If the intersection is empty, the function reports the minimum feasible epsilon and raises; it does not silently relax it. Changing epsilon after inspecting results is another selection choice.


In [ ]:
STEP = 0.05
EPSILON = 0.05
selected = learn_common_weights(
    panel.y, oof, panel.train.day_id,
    repeats=4, n_splits=5, seed=42, step=STEP, epsilon=EPSILON,
)
display(selected.weights.rename("synthetic_weight"))
print("Minimum feasible epsilon on this grid:", selected.min_feasible_epsilon)
print("Selected full-OOF score (not independent):", selected.selection_score)
display(selected.candidates.loc[selected.candidates.common].sort_values("full_oof", ascending=False).head(10))
display(selected.diagnostics)


## 3. A separate second-stage diagnostic

Here the blend weight is selected on four partitions and scored on the fifth. Held-out labels do not select that split's weight. This remains conditional on existing base OOF and previously selected models; it is **not** end-to-end nested cross-validation.


In [ ]:
cross_fitted = cross_fitted_weight_score(
    panel.y, oof, panel.train.day_id, n_splits=5, seed=42, step=STEP,
)
display(cross_fitted)
print("Conditional cross-fitted blender score:", cross_fitted.attrs["pooled_score"])
equal_weight_score = weighted_directional_accuracy(panel.y, oof.mean(axis=1))
print("Equal-weight synthetic baseline:", equal_weight_score)


## 4. Apply the selected weights to full-refit test predictions

Use exactly the same selected weights for OOF diagnostics and full-refit raw test predictions. Threshold raw values at zero to create a local synthetic submission-style file. No external submission occurs.


In [ ]:
exported = run_ensemble(panel, ARTIFACT_DIR, epsilon=EPSILON)
pd.testing.assert_series_equal(exported.weights, selected.weights)
print("Synthetic artifacts saved under:", (ARTIFACT_DIR / "ensemble").name)


## Limits

A finite grid optimum is not a continuous global optimum. Near-optimal-set overlap is a stability diagnostic, not proof of independent generalization. The selected full-OOF score uses labels consulted during selection. Competition performance evidence belongs in the separate [results record](../docs/RESULTS.md), not in these synthetic numbers.
